## 1.- Transfer Learning y Fine Tuning para clasificación de imágenes

En este apartado, vamos a tomar el dataset 101 Object, que consta de 6907 imágenes de entrenamiento y 1770 de test, pertenecientes a 101 clases diferentes.

Las imágenes son bastante complejas, con una resolución muy alta, y además contamos con, relativamente, pocos datos por cada clase - unas 70 imágenes de entrenamiento y 17 de test por clase -, lo que hace que pueda ser difícil obtener buenos resultados entrenando un modelo desde 0.

Por ello, la finalidad de esta tarea es conseguir el mejor resultado posible utilizando técnicas de Transfer Learning y Fine Tuning. Para limitar el alcance del problema, vamos a limitar el uso de modelos preentrenados, pudiendo utilizar solo el modelo VGG16.

Para poner más dificultades al modelo e intentar exprimir al máximo las herramientas de Fine Tuning y Transfer Learning, **no vamos a permitir utilizar técnicas de Data Augmentation.**

Además del desarrollo del código, se pide responder a las siguientes preguntas:

- 1.- ¿Cuáles son los motivos principales por los que entrenar un modelo desde 0 no sería lo ideal en este caso?
- 2.- ¿Qué tendría que cambiar para que pudieramos entrenar un modelo desde 0 para resolver este problema?
- 3.- Si quisieramos un modelo capaz de ser entrenado lo más rápidamente posible a cambio de poder tener un peor rendimiento, ¿cómo lo haríamos en este caso?

*Nota: aquí nos estamos enfrentando a un problema muy complicado. La falta de datos y el elevado número de clases va a hacer que los accuracies que consigamos sean muy bajos. No os preocupéis por ello, el reto de esta tarea reside aquí.*

La siguiente celda es para leer los datos desde Drive si utilizamos Google Colab. Si no utilizamos esta herramienta, no debemos ejecutar esta celda.

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

La siguiente celda tiene la ruta al dataset utilizado. Si se ha guardado directamente la carpeta con los materiales *Máster DL - IEP* en nuestra página principal de Drive, debería funcionar sin tocar nada.

In [3]:
# Directorio que contiene las carpetas de entrenamiento y prueba
PATH = "dataset/101_ObjectCategories"

Ahora, pasamos a leer los datos. Esta celda se explica paso a paso, pero no es fundamental para el proyecto. Lo único que estamos haciendo es obtener los data flows desde las carpetas directamente con ayuda de la clase *ImageDataGenerator*. El método *flow_from_directory* nos permite directamente utilizar los datos, asignando a cada carpeta una clase/etiqueta diferente, lo cuál es útil siempre que tengamos una estructura similar.

También aprovechamos a configurar los parámetros para llevar a cabo Data Augmentation. **Estos parámetros NO se pueden modificar** y todos debemos utilizar los mismos.

In [4]:
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configurar los generadores de datos para entrenamiento y prueba
train_datagen = ImageDataGenerator(rescale=1./255)

test_datagen = ImageDataGenerator(rescale=1./255)  # Normalización de píxeles

# Especificar cómo leer los datos de entrenamiento y prueba desde el directorio
train_generator = train_datagen.flow_from_directory(
        directory= PATH + '/train',
        target_size = (224, 224),  # Tamaño de las imágenes (ajústalo según sea necesario)
        batch_size = 32,
        shuffle=True,
        class_mode = 'categorical')  # Modo de clasificación categórica

test_generator = test_datagen.flow_from_directory(
        directory = PATH + '/val',
        target_size = (224, 224),  # Tamaño de las imágenes (ajústalo según sea necesario)
        batch_size = 16,
        shuffle=True,
        class_mode = 'categorical')  # Modo de clasificación categórica

Found 6878 images belonging to 101 classes.
Found 1770 images belonging to 101 classes.


### Entrenando un modelo desde cero

Primero, vamos a generar un modelo desde 0 y entrenarlo con los datos. Aunque podríamos probar con redes más complejas, va a ser difícil conseguir un accuracy bueno. Podemos ver en el siguiente ejemplo, donde obtenemos alrededor de un 50% de accuracy (dependiendo de la iteración).

In [5]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Definir el modelo de la CNN
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(101, activation='softmax'))

# Compilar el modelo
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(
      train_generator,
      steps_per_epoch=train_generator.samples // train_generator.batch_size,
      epochs=20,
      validation_data=test_generator,
      validation_steps=test_generator.samples // test_generator.batch_size)

# Evaluar el modelo
test_loss, test_acc = model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

c:\Users\nunoc\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\nunoc\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 492s 2s/step - accuracy: 0.1856 - loss: 4.5397 - val_accuracy: 0.3977 - val_loss: 2.8719
Epoch 2/20
  1/214 ━━━━━━━━━━━━━━━━━━━━ 6:30 2s/step - accuracy: 0.4062 - loss: 3.1005

c:\Users\nunoc\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


214/214 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - accuracy: 0.4062 - loss: 3.1005 - val_accuracy: 0.3955 - val_loss: 2.8663
Epoch 3/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 350s 2s/step - accuracy: 0.4834 - loss: 2.3747 - val_accuracy: 0.4756 - val_loss: 2.3305
Epoch 4/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 20s 87ms/step - accuracy: 0.5312 - loss: 2.1629 - val_accuracy: 0.4841 - val_loss: 2.2814
Epoch 5/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 355s 2s/step - accuracy: 0.6916 - loss: 1.2872 - val_accuracy: 0.5006 - val_loss: 2.3385
Epoch 6/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 21s 93ms/step - accuracy: 0.8125 - loss: 0.9718 - val_accuracy: 0.5057 - val_loss: 2.3313
Epoch 7/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 345s 2s/step - accuracy: 0.8443 - loss: 0.6017 - val_accuracy: 0.5295 - val_loss: 2.5416
Epoch 8/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 21s 91ms/step - accuracy: 0.8750 - loss: 0.5859 - val_accuracy: 0.5256 - val_loss: 2.5379
Epoch 9/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 345s 2s/step - accuracy: 0.9136 - loss: 0.3470 - val_accuracy: 0

## Transfer Learning

Ahora, pasamos a entrenar llevando a cabo un ejercicio de Transfer Learning. Para ello, vamos a leer el modelo VGG16 con los pesos del modelo entrenado con imagenet y quitándole las últimas capas, vamos a congelar las capas de este modelo y vamos a añadir las últimas capas para la clasificación.

In [7]:
from keras.applications import VGG16
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout
from keras.optimizers import Adam

# Cargar el modelo VGG16 pre-entrenado sin la capa de clasificación
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Congelar las capas del modelo base
for layer in base_model.layers:
    layer.trainable = False

# Definir el modelo secuencial sobre el modelo base
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(101, activation='softmax'))


# Compilar el modelo
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(
      train_generator,
      steps_per_epoch=train_generator.samples // train_generator.batch_size,
      epochs=20,
      validation_data=test_generator,
      validation_steps=test_generator.samples // test_generator.batch_size)

# Evaluar el modelo
test_loss, test_acc = model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 1106s 5s/step - accuracy: 0.3705 - loss: 3.1495 - val_accuracy: 0.7352 - val_loss: 1.4067
Epoch 2/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.6250 - loss: 1.6560 - val_accuracy: 0.7375 - val_loss: 1.3994
Epoch 3/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 1112s 5s/step - accuracy: 0.7402 - loss: 1.2174 - val_accuracy: 0.8250 - val_loss: 0.8606
Epoch 4/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.9062 - loss: 0.6280 - val_accuracy: 0.8256 - val_loss: 0.8682
Epoch 5/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 1126s 5s/step - accuracy: 0.8618 - loss: 0.6617 - val_accuracy: 0.8659 - val_loss: 0.6470
Epoch 6/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 229s 1s/step - accuracy: 0.7500 - loss: 0.8484 - val_accuracy: 0.8619 - val_loss: 0.6492
Epoch 7/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 1123s 5s/step - accuracy: 0.9139 - loss: 0.4253 - val_accuracy: 0.8784 - val_loss: 0.5606
Epoch 8/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.9688 - loss: 0.3226 - val_

## Fine Tuning

Por último, vamos a llevar a cabo un ejercicio de Fine Tuning. Para ello, prácticamente podemos calcar el código utilizado anteriormente, con la única diferencia de que no deberemos congelar ninguna capa de la red.

In [ ]:
# TÚ CÓDIGO AQUÍ
from keras.applications import VGG16
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout
from keras.optimizers import Adam

# Cargar el modelo VGG16 completo, sin quitar las capas convolucionales
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# No congelamos las capas

# Crear el modelo completo
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(101, activation='softmax'))

# Compilar el modelo
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=20,
    validation_data=test_generator,
    validation_steps=test_generator.samples // test_generator.batch_size
)

# Evaluar el modelo
test_loss, test_acc = model.evaluate(test_generator, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 3194s 15s/step - accuracy: 0.3245 - loss: 3.2987 - val_accuracy: 0.7159 - val_loss: 1.1949
Epoch 2/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.7500 - loss: 1.1570 - val_accuracy: 0.7119 - val_loss: 1.2369
Epoch 3/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 3207s 15s/step - accuracy: 0.7267 - loss: 1.1542 - val_accuracy: 0.8080 - val_loss: 0.7999
Epoch 4/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 256s 1s/step - accuracy: 0.8125 - loss: 0.6758 - val_accuracy: 0.8023 - val_loss: 0.8168
Epoch 5/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 3253s 15s/step - accuracy: 0.8624 - loss: 0.5479 - val_accuracy: 0.8278 - val_loss: 0.6590
Epoch 6/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.9688 - loss: 0.2091 - val_accuracy: 0.8290 - val_loss: 0.6531
Epoch 7/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 3195s 15s/step - accuracy: 0.9242 - loss: 0.2924 - val_accuracy: 0.8580 - val_loss: 0.5820
Epoch 8/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 249s 1s/step - accuracy: 0.9375 - loss: 0.1601 - 

Conclusión: En este ejercicio, trabajamos con un problema de clasificación de imágenes usando el dataset 101 Object Categories. Probamos tres enfoques distintos: entrenar un modelo desde cero, usar transfer learning con VGG16 congelada y luego hacer fine tuning con VGG16 completa.

Primero, al entrenar una red CNN desde cero, el modelo alcanzó una precisión de test de aproximadamente 53%, lo cual es aceptable considerando la dificultad del problema, pero mostró señales de sobreajuste. Luego, con transfer learning congelando las capas de VGG16, el resultado mejoró muchísimo: obtuvimos casi 90% de precisión, en mucho menos tiempo. Finalmente, con fine tuning, donde entrenamos todas las capas de VGG16, logramos un 85% de precisión, pero con tiempos de entrenamiento mucho más largos.

La mejor opción fue usar transfer learning congelando las capas, ya que fue más rápido y preciso.

## 2.- Prompt Engineering

En este apartado, se pide desarrollar una serie de prompts, utilizando las técnicas de prompt engineering necesarias para obtener los mejores resultados posibles.

Para esta serie de ejercicios se recomienda utilizar [ChatGPT](https://chatgpt.com/) desde su interfaz web, ya que a día de hoy es de uso libre y no tiene muchas restricciones, pero se puede utilizar cualquier otro modelo.

### Ejercicio 2.1

Imaginemos que estamos trabajando en una consultora tecnológica especializada en datos - data science, datga engineering, arquitectura de Big Data, etc - y que queremos utilizar algún LLM para generar automáticamente ofertas que poder lanzar a los clientes en base a cierta información.

Así, queremos una plantilla de un prompt que, simplemente cambiando los perfiles, el tiempo de ejecución, y precio del proyecto, genere las ofertas para poder lanzarlas al cliente.

4.- ¿Qué técnica o técnicas de prompt engineering estaríamos utilizando aquí?

R: En este caso, estaríamos utilizando principalmente la técnica de "prompt templating", ya que construimos una plantilla fija donde simplemente cambiamos algunos campos según la necesidad (perfil, duración y precio). Además, aplicamos "parameter injection", ya que los datos específicos se introducen en la estructura del prompt automáticamente. Si separamos el prompt en bloques reutilizables, podríamos hablar también de "prompt modularity".

### Ejercicio 2.2

Queremos automatizar una herramienta para el departamento de recursos humanos que agilice la revisión de curriculumns. Para ello, queremos un prompt que nos ayude a resumir y extraer la información relevante para el puesto. Por ejemplo, si el candidato tiene experiencia como médico pero está aplicando a un puesto de programador, no nos interesará que en el resumen aparezca ese tipo de información.

Diseña un prompt capaz de obtener la información relevante para un puesto de data scientist para un proyecto relacionado con el comercio internacional para la Secretaria de Estado del Comercio. Aplica el prompt al siguiente curriculum para afinar y obtener el prompt definitivo.


------------------------------------------------------------------------
Curriculum Vitae

Nombre: [Nombre del Candidato]

Información de Contacto:
Dirección: [Dirección]
Teléfono: [Número de Teléfono]
Correo Electrónico: [Correo Electrónico]

Perfil Profesional:

Abogado con una sólida experiencia en el ámbito legal, respaldado por una carrera de varios años en diferentes bufetes de abogados. Recientemente, ha complementado su formación con un máster en Big Data y cursos adicionales en programación e idiomas. Posee habilidades analíticas y técnicas avanzadas, combinadas con un profundo conocimiento del marco legal. Apasionado por la resolución de problemas y la optimización de procesos mediante el uso de tecnología y datos.

Educación:

Máster en Big Data - [Nombre de la Universidad / Institución], [Año de Graduación]
Licenciatura en Derecho - [Nombre de la Universidad], [Año de Graduación]


Experiencia Laboral:

Bufete de Abogados XYZ
Posición: Abogado Senior
Periodo: [Fecha de Inicio] - [Fecha de Finalización]

Responsabilidades:
Liderazgo en casos complejos de litigación civil y penal.
Asesoramiento legal a clientes corporativos en cuestiones regulatorias y de cumplimiento.
Negociación y redacción de contratos comerciales.
Gestión de equipos y supervisión de pasantes legales.


Bufete de Abogados ABC
Posición: Abogado Asociado
Periodo: [Fecha de Inicio] - [Fecha de Finalización]

Responsabilidades:
Investigación legal y preparación de argumentos para casos judiciales.
Representación de clientes en procedimientos administrativos.
Elaboración de informes legales y dictámenes jurídicos.
Colaboración con otros departamentos para garantizar el cumplimiento normativo.


Habilidades:

Análisis de datos y modelado predictivo.
Programación en Python, SQL y R.
Manejo avanzado de herramientas de Big Data (Hadoop, Spark, etc.).
Conocimientos legales en áreas como derecho civil, penal y corporativo.
Excelentes habilidades de comunicación y negociación.
Dominio de idiomas: [Idiomas]


Intereses y Actividades Extracurriculares:

Tocar la guitarra: Apasionado por la música, dedica parte de su tiempo libre a la práctica y el estudio de la guitarra.
Voluntariado: Participa activamente en proyectos de voluntariado comunitario, brindando asesoramiento legal pro bono a personas necesitadas.


Referencias:

Disponibles a solicitud.

5.- ¿Qué tipo de técnicas de prompt engineering se han probado para elaborar el prompt?¿Cuáles han funcionado mejor?

R: En este caso se utiliza la técnica de "role prompting", ya que se le indica al modelo que actúe como un asistente de recursos humanos especializado en revisión de currículums. También aplicamos instrucciones explícitas para enfocar la salida solo en la información relevante para un puesto específico. Además, se utiliza prompt enfocado en tarea (task specification) para guiar claramente lo que debe hacer y lo que debe ignorar. Por último, se utiliza estructura de salida (output structuring), al pedir que la respuesta esté organizada en bloques concretos (formación, habilidades, idiomas, experiencia).

El prompt utilizado:

Actúa como reclutador especializado en perfiles de ciencia de datos.

Tu tarea es analizar el currículum y generar un resumen enfocado únicamente en la información relevante para un puesto de Data Scientist en un proyecto de comercio internacional para la Secretaría de Estado del Comercio.

Ignora la experiencia laboral legal o que no esté relacionada con ciencia de datos, análisis, programación o comercio.  
Prioriza habilidades técnicas, formación en Big Data o programación, idiomas relevantes y experiencia aplicable al entorno de datos.

Organiza el resumen en 4 apartados:
1. Formación técnica relevante  
2. Habilidades técnicas útiles  
3. Idiomas  
4. Experiencia aplicable al análisis de datos y comercio internacional

Aquí tienes el currículum:

[Inserir currículo aqui]

Respuesta de la IA:

1. Formación técnica relevante:
- Máster en Big Data
- Cursos de programación e idiomas

2. Habilidades técnicas útiles:
- Análisis de datos y modelado predictivo
- Programación en Python, SQL y R
- Manejo avanzado de herramientas de Big Data como Hadoop y Spark

3. Idiomas:
- [Idiomas] (potencialmente útiles para comercio internacional)

4. Experiencia aplicable:
- Aunque su experiencia principal es legal, ha desarrollado habilidades analíticas y de optimización de procesos mediante tecnología y datos, lo cual puede ser valioso en un entorno de análisis de comercio internacional.